# Sphere benchmark: paired statistics (supplement to the violins)

Data: `results/correction_benchmark/sphere_paired_stats.csv` (`src/correction_benchmark/sphere_paired_stats.py`).
Per graph type x measure, BOSPERRUS (additive) against each other method, paired by point set
(3 patterns x 100 seeds; k-NN / r-NN graphs on one point set averaged). Cell: median paired difference in
Spearman rho and the fraction of point sets where BOSPERRUS wins; colour: verdict of one-sided Wilcoxon
signed-rank tests, Holm-corrected, margin 0.01.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from matplotlib.transforms import blended_transform_factory

TEXTWIDTH = (8.5 * 2.54 - 4.0) / 2.54  # letterpaper, 2 cm margins
plt.rcParams.update({"pdf.fonttype": 42, "svg.fonttype": "none", "font.size": 6.5})

stats = pd.read_csv("../../results/correction_benchmark/sphere_paired_stats.csv")
GRAPHS = {"delaunay": "Delaunay", "knn": "$k$-NN", "rnn": "$r$-NN"}
MEASURES = {"betweenness": "betweenness", "closeness": "closeness", "clustering": "clustering",
            "degree": "degree", "pagerank": "PageRank"}
COMPETITORS = {"raw": "raw", "spline": "spline detr.", "linear": "linear detr.", "SERN": "SERN"}
VERDICT_COLORS = {"better": "#8ABBD6", "non-inferior": "#E6E6E6", "worse": "#F4A582", "inconclusive": "#FFFFFF"}
VERDICT_LABELS = {"better": "better (by > 0.01)", "non-inferior": "not worse by > 0.01",
                  "worse": "worse by > 0.01", "inconclusive": "inconclusive"}
rows = [(g, m) for g in GRAPHS for m in MEASURES]


def panel(ax):
    s = stats[stats["bosperrus"] == "bosperrus"].set_index(["graph_type", "measure", "competitor"])
    for i, (g, m) in enumerate(rows):
        for j, c in enumerate(COMPETITORS):
            r = s.loc[(g, m, c)]
            ax.add_patch(plt.Rectangle((j, i), 1, 1, color=VERDICT_COLORS[r["verdict"]], lw=0))
            ax.text(j + 0.5, i + 0.5, f"{r['median_delta']:+.2g}\n{r['win_rate'] * 100:.0f} %", ha="center",
                    va="center", fontsize=5, linespacing=1.1)
    for i in range(len(MEASURES), len(rows), len(MEASURES)):
        ax.axhline(i, color="k", lw=0.6)
    ax.set_xlim(0, len(COMPETITORS))
    ax.set_ylim(len(rows), 0)
    ax.set_xticks(np.arange(len(COMPETITORS)) + 0.5, list(COMPETITORS.values()))
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    ax.set_yticks(np.arange(len(rows)) + 0.5, [MEASURES[m] for _, m in rows])
    for k, g in enumerate(GRAPHS):
        ax.text(0.025, (k + 0.5) * len(MEASURES), GRAPHS[g], rotation=90, ha="center", va="center",
                fontweight="bold", transform=blended_transform_factory(ax.figure.transFigure, ax.transData))
    ax.set_title("BOSPERRUS vs.", fontsize=6.5, pad=14)
    ax.spines[:].set_visible(False)


fig, ax = plt.subplots(figsize=(TEXTWIDTH * 0.6, TEXTWIDTH * 0.62))
fig.subplots_adjust(left=0.27, right=0.98, top=0.9, bottom=0.14)
panel(ax)
fig.legend([Patch(color=VERDICT_COLORS[v], ec="0.6", lw=0.3) for v in VERDICT_COLORS], VERDICT_LABELS.values(),
           loc="lower center", ncol=2, frameon=False, fontsize=6, title="BOSPERRUS is ... (Holm-corrected, α = 0.05)",
           title_fontsize=6)
fig.savefig("sphere_paired_stats.svg", dpi=300)